In [0]:
%run ./01_logging_utils.py

In [0]:
import time

TABLES = [T_FACT_HOURLY, T_FACT_DAILY, T_FACT_PORTF]
ZCOLS  = {T_FACT_HOURLY: "coin_key, date_key", T_FACT_DAILY: "coin_key, date_key", T_FACT_PORTF: "date_key"}

bk = spark.sql("SELECT xxhash64('bitcoin')").first()[0]
r  = spark.sql(f"SELECT date_key FROM {T_DIM_DATE} ORDER BY date_key DESC LIMIT 1 OFFSET 7").first()
dk = r[0] if r else 0
Q = (f"SELECT count(*) n, avg(price) p, max(hourly_return_pct) r FROM {T_FACT_HOURLY} "
     f"WHERE coin_key = {bk} AND date_key >= {dk}")

def bench(n=3):
    ts = []
    for _ in range(n):
        s = time.time(); spark.sql(Q).collect(); ts.append(time.time() - s)
    return round(min(ts), 3)

def snap(t):
    d = spark.sql(f"DESCRIBE DETAIL {t}").first()
    return d["numFiles"], d["sizeInBytes"], list(d["clusteringColumns"] or [])

rows = []
for t in TABLES:
    f0, b0, cl = snap(t)
    q0 = bench() if t == T_FACT_HOURLY else None
    rid, t0 = new_batch_id("run"), datetime.now()
    try:
        spark.sql(f"OPTIMIZE {t}" if cl else f"OPTIMIZE {t} ZORDER BY ({ZCOLS[t]})")
        status, err = "SUCCESS", None
    except Exception as e:
        status, err = "FAILED", str(e)[:500]
    f1, b1, _ = snap(t)
    q1 = bench() if t == T_FACT_HOURLY else None
    log_run(rid, "gold",
            f"OPTIMIZE {t} | files {f0}->{f1} | bytes {b0}->{b1} | query_s {q0}->{q1} | clustering={cl}",
            t0, datetime.now(), status, 0, 0, err)
    rows.append((t, ",".join(cl) or "none (ZORDER used)", int(f0), int(f1), int(b0), int(b1), q0, q1, status))

display(spark.createDataFrame(rows,
    "table string, clustering string, files_before int, files_after int, "
    "bytes_before long, bytes_after long, query_s_before double, query_s_after double, status string"))